In [ ]:
# Written by tools/build_notebooks.py -- do not edit. The bootstrap cell below
# compares this against the repo it clones and tells you if these cells are old.
CELLS_SRC = "kaggle_02_train_counter.py"
CELLS_SHA = "058379830d415be3"

# 02 — Train the counter

**Accelerator: GPU T4 ×2.** **Runtime: 2–4 hours.** Costs roughly **3 GPU-hours** of your
30 per week.

## This notebook has to earn its place

Notebook 01 already counted speakers on CPU. This one only matters if it **beats** that
number. If it does not, that is a result worth reporting, not a failure to hide — and the
training log says so on every line.

It trains the counter **alone**, which is the central change in v1. In the old project a
counting head hung off the separator's trunk and received **0.53 %** of the gradient while
separation took 86.14 % — 633:1 at the encoder. It never learned, and settled on answering
"1 speaker" for 1445 of 1500 test mixtures. Here counting gets 100 % of its own gradient.

## The one thing that actually matters here

It is not the architecture, the learning rate, or the amount of data. It is **how the model
summarises time**. Measured on identical data, changing *only* that one layer:

| how it summarises | accuracy |
|---|---|
| mean + standard deviation (what the old project used) | 66.5 % |
| attention-weighted mean + std | 67.5 % |
| the full covariance matrix | **20.0 % — collapsed to "1 speaker"** |
| **the covariance's eigenvalue spectrum** (the default here) | **74.4 %** |

An 8-point swing from one layer. The winner is also the *smallest* and *fastest* of the four.

The intuition: if N people are talking, the sound occupies roughly **N independent
directions**. Eigen*values* tell you **how many** directions carry energy — that is the count.
Eigen*vectors* tell you **which** directions — and a counter should not care which. Throwing
the eigenvectors away is not a shortcut; it is the whole idea.

## Before you press Run

1. **Settings → Accelerator → GPU T4 ×2**
2. **+ Add Input → Notebook Output →** notebook 00's output (the store)
3. **+ Add Input → Notebook Output →** notebook 01's output (for the bar)
4. **Settings → Internet → On** (only if the code comes from GitHub)
5. **Settings → Persistence → Variables and Files** — so a 12-hour timeout does not lose
   your checkpoints

## Bootstrap (this cell is identical in every notebook)

Three ways to get the code onto the Kaggle machine, tried in order:

1. **GitHub clone** — set `REPO_URL` below and turn *Internet* ON in the notebook
   settings panel (Settings → Internet → On). This is the recommended route.
2. **Repo-as-dataset** — upload this folder as a Kaggle Dataset called
   `speaker-count-separate-v1` and attach it. No internet needed. Use this if your
   account cannot enable internet (phone-verification is required for that).
3. **Already there** — an existing clone is **fast-forwarded to the newest commit**,
   not reused as-is. A Kaggle session outlives many pushes, and silently running code
   from an hour ago is the most expensive kind of confusion: the log looks fine and the
   fix you are testing is not in it. Any local edits inside the clone are discarded.

Whichever route runs, the commit is printed. Every log can then be traced to the exact
code that produced it.

In [ ]:
REPO_URL = "https://github.com/AlAminAshraf01/speaker-count-separate-v1.git"
REPO_DIR = "/kaggle/working/speaker-count-separate-v1"
REPO_AS_DATASET = "/kaggle/input/speaker-count-separate-v1"

import hashlib
import os
import shutil
import subprocess
import sys


def cells_fingerprint(src_dir: str, name: str) -> str:
    """Short hash of one notebook's percent source plus this shared bootstrap.

    ``tools/build_notebooks.py`` stamps this into every generated ``.ipynb``. The copy
    running on Kaggle recomputes it from the freshly-cloned repo, so a notebook whose
    cells were imported before the last push says so in the first ten seconds instead of
    eleven hours later.

    Line endings are normalised first. The same file is CRLF in a Windows working tree
    and LF in a Linux clone, and a fingerprint that disagrees with itself across
    platforms is worse than no fingerprint at all.
    """
    digest = hashlib.sha256()
    for part in (name, "_bootstrap.py"):
        with open(os.path.join(src_dir, part), "rb") as fh:
            digest.update(fh.read().replace(b"\r\n", b"\n"))
        digest.update(b"\0")
    return digest.hexdigest()[:16]


def cells_status(repo_dir: str, src_name: str | None, stamp: str | None) -> str:
    """Compare the stamp baked into these cells with the repo they are about to run.

    Never raises. A check that can take down every notebook is a worse bug than the one
    it detects, so anything unreadable degrades to "cannot verify".
    """
    if not src_name or not stamp:
        return "unstamped -- re-import this notebook to enable the staleness check"
    try:
        current = cells_fingerprint(os.path.join(repo_dir, "notebooks", "src"), src_name)
    except Exception as exc:
        return f"cannot verify ({exc})"
    if current == stamp:
        return f"current ({stamp})"
    return "\n".join([
        f"STALE  cells {stamp} but repo has {current}",
        "",
        "  These notebook cells were imported before the newest push, so the fix you",
        "  are about to test is not in them. scripts/ and src/ just updated themselves;",
        "  notebook cells cannot, because Kaggle owns them.",
        "",
        "  Fix: File -> Import Notebook -> upload notebooks/" + src_name[:-3] + ".ipynb",
        "       again, re-attach the inputs, and re-run.",
    ])


def _git(repo_dir: str, *argv: str) -> subprocess.CompletedProcess:
    return subprocess.run(["git", "-C", repo_dir, *argv],
                          capture_output=True, text=True)


def update_clone(repo_dir: str) -> str:
    """Fast-forward an existing clone to the remote's newest commit.

    Returns a short status for printing; never raises. Losing internet is a reason to
    carry on with the code that is already there, but it is not a reason to be quiet
    about it -- running stale code unknowingly is how a fix gets tested without being
    present.
    """
    if not os.path.isdir(os.path.join(repo_dir, ".git")):
        return "not a git clone, left as it is"
    branch = _git(repo_dir, "rev-parse", "--abbrev-ref", "HEAD").stdout.strip() or "main"
    before = _git(repo_dir, "rev-parse", "--short", "HEAD").stdout.strip()
    fetched = _git(repo_dir, "fetch", "--depth", "1", "origin", branch)
    if fetched.returncode != 0:
        tail = (fetched.stderr or "").strip().splitlines()
        return f"COULD NOT FETCH ({tail[-1] if tail else 'unknown'}) -- code may be stale"
    reset = _git(repo_dir, "reset", "--hard", f"origin/{branch}")
    if reset.returncode != 0:
        tail = (reset.stderr or "").strip().splitlines()
        return f"COULD NOT UPDATE ({tail[-1] if tail else 'unknown'}) -- code may be stale"
    after = _git(repo_dir, "rev-parse", "--short", "HEAD").stdout.strip()
    return "already newest" if before == after else f"updated {before} -> {after}"


def describe_commit(repo_dir: str) -> str:
    """``<short sha> <date> <subject>`` for the checked-out commit, or a plain note."""
    out = _git(repo_dir, "log", "-1", "--format=%h %cs %s").stdout.strip()
    return out or "no git metadata"


def bootstrap(repo_url: str = REPO_URL, repo_dir: str = REPO_DIR) -> str:
    """Put the repo at `repo_dir`, put its `src/` on sys.path, and chdir into it."""
    if not os.path.isdir(os.path.join(repo_dir, "src")):
        if os.path.isdir(os.path.join(REPO_AS_DATASET, "src")):
            shutil.copytree(REPO_AS_DATASET, repo_dir, dirs_exist_ok=True)
            print(f"copied repo from the attached dataset {REPO_AS_DATASET}")
        else:
            subprocess.run(["git", "clone", "--depth", "1", repo_url, repo_dir], check=True)
            print(f"cloned {repo_url}")
    else:
        print(f"existing clone: {update_clone(repo_dir)}")
    src = os.path.join(repo_dir, "src")
    if src not in sys.path:
        sys.path.insert(0, src)
    os.chdir(repo_dir)
    return repo_dir


REPO = bootstrap()

import countsep  # noqa: E402

print("countsep", countsep.__version__, "at", REPO)
print("code ", describe_commit(REPO))
# CELLS_SRC / CELLS_SHA are set by the stamp cell that tools/build_notebooks.py puts at
# the top of every generated notebook. globals().get keeps this working in a notebook
# assembled by hand, where that cell may not exist.
CELLS = cells_status(REPO, globals().get("CELLS_SRC"), globals().get("CELLS_SHA"))
print("cells", CELLS if "\n" not in CELLS else "")
if "\n" in CELLS:
    print(CELLS)
print("python", sys.version.split()[0])

import torch  # noqa: E402

print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| devices", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  [{i}] {p.name}  {p.total_memory / 1e9:.1f} GB")

In [ ]:
import shlex
import time


def run(cmd: str, check: bool = True) -> int:
    """Run a shell command, streaming its output into the notebook."""
    print("$", cmd, flush=True)
    t0 = time.time()
    proc = subprocess.Popen(shlex.split(cmd), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="", flush=True)
    code = proc.wait()
    print(f"\n[exit {code} in {time.time() - t0:.1f}s]", flush=True)
    if check and code != 0:
        raise SystemExit(f"command failed with exit code {code}")
    return code

In [ ]:
import sys
sys.path.insert(0, os.path.join(REPO, "scripts"))
from _common import autodetect_ckpt, autodetect_store, find_recipes

STORE = autodetect_store()
RECIPES_DEV = find_recipes("recipes_dev.csv", STORE)
OUT = "/kaggle/working/counter"
POOLING = "eigen"        # meanstd | attentive | covariance | eigen

print("store      :", STORE)
print("recipes dev:", RECIPES_DEV)
if STORE is None:
    raise SystemExit("No packed store. '+ Add Input' -> 'Notebook Output' -> notebook 00.")

## Find the bar from notebook 01

If notebook 01's output is attached, its number is used automatically. Otherwise the default
is the measured 69.3 %, which is close enough to be a fair target.

In [ ]:
import glob
import json

BAR = 0.693
for path in glob.glob("/kaggle/input/**/tier_a_report.json", recursive=True):
    with open(path) as fh:
        BAR = float(json.load(fh)["bar"])
    print(f"found Tier A report: {path}")
    break
else:
    print("no Tier A report attached; using the measured default")
print(f"the bar to beat: {BAR:.1%}")

## Check before you spend  (~30 s)

The row to watch is **precision**.

The old counter scored **44.9 %** under fp16 and **20.00 %** in fp32 *from the same saved
model* — they agreed on only 16 % of their answers. It had been trained in one kind of
arithmetic and tested in the other, so the thing that was measured was never the thing that
was trained. Nobody noticed for weeks, because both numbers look like plausible accuracies.

Here that is a thirty-second check that stops the notebook, instead of a discovery you make
after eleven hours of GPU time.

In [ ]:
run(f"python scripts/preflight.py --for train --store {STORE}"
    + (f" --recipes_dev {RECIPES_DEV}" if RECIPES_DEV else "")
    + f" --cells_src {CELLS_SRC} --cells_sha {CELLS_SHA}")

## Train  (~2–3 h)

Notes on the settings, since you may want to change them:

* **`--amp` is off.** Mixed precision would make this maybe 30 % faster and reopens exactly
  the failure described above. The model is small; the trade is not worth it. If you do turn
  it on, the preflight check and the end-of-training check both still run.
* **It resumes.** If the 12-hour limit kills the session, re-run this same cell — it picks up
  from the last checkpoint rather than starting over.
* **`--time_budget_h 10.5`** stops it cleanly before Kaggle's hard cut-off.
* Every epoch prints `beats the bar` or `BELOW THE BAR`. Watch that column, not the loss.

In [ ]:
run(f"python scripts/04_train_counter.py"
    f" --store {STORE}"
    + (f" --recipes_dev {RECIPES_DEV}" if RECIPES_DEV else "")
    + f" --pooling {POOLING}"
    f" --epochs 20"
    f" --steps_per_epoch 2000"
    f" --batch_size 64"
    f" --lr 2e-3"
    f" --num_workers 3"
    f" --bar {BAR:.4f}"
    f" --time_budget_h 10.5"
    f" --out {OUT}")

## Optional — compare the four pooling layers yourself

This is the measurement that produced the table at the top. Four short runs, about
**1.5 GPU-hours** in total. Worth doing if you want the comparison in your own report with
your own data; skip it if quota is tight.

Expect `covariance` to collapse to 20 %. That is the expected result, not a bug — and it is
the most interesting row in the table, because it is the same idea as `eigen` implemented
the obvious way.

In [ ]:
COMPARE_POOLINGS = False        # set True to run it

if COMPARE_POOLINGS:
    for pooling in ("meanstd", "attentive", "covariance", "eigen"):
        run(f"python scripts/04_train_counter.py"
            f" --store {STORE}"
            + (f" --recipes_dev {RECIPES_DEV}" if RECIPES_DEV else "")
            + f" --pooling {pooling}"
            f" --epochs 6 --steps_per_epoch 1000 --batch_size 64"
            f" --bar {BAR:.4f} --time_budget_h 2.0"
            f" --out /kaggle/working/pool_{pooling}")

## Read the verdict

The final block of the training log says either `worth keeping` or `BOUGHT NOTHING`. Both are
legitimate outcomes and both belong in your report.

**Save Version → Save & Run All (Commit)** so the checkpoint becomes a dataset for notebook 03.

In [ ]:
with open(os.path.join(OUT, "train_report.json")) as fh:
    report = json.load(fh)

print(f"pooling ................. {report['pooling']}")
print(f"parameters .............. {report['params'] / 1e6:.3f} M")
print(f"best dev accuracy ....... {report['best_val_accuracy']:.1%}")
print(f"the bar (Tier A) ........ {report['bar']:.1%}")
print(f"beats the bar ........... {report['beats_bar']}")
print(f"fp32/fp16 agreement ..... {report['precision_agreement']:.1%}")
print()
print("Nothing here is a final number. These are DEV numbers, used to choose things.")
print("The test set is opened once, in notebook 04, and not before.")